In [2]:
import random
from pathlib import Path

class Loot:
  def __init__(self, kv_probs: dict):
    self.__kv_probs = kv_probs
    self.__total = sum(kv_probs.values())
    self.__gen = random.Random()

  def next(self):
    chosen = self.__gen.randint(1, self.__total)
    acc = 0

    for item, weight in self.__kv_probs.items():
      acc += weight
      if chosen <= acc:
        return item

animal_weights = {
    "souris": 50,
    "pigeon": 40,
    "écureuil": 35,
    "chat": 30,
    "chien": 25,
    "lapin": 20,
    "renard": 15,
    "chouette": 12,
    "cerf": 10,
    "sanglier": 8,
    "loup": 5,
    "aigle": 4,
    "ours": 3,
    "lynx": 2,
    "panthère": 1,
    "phénix": 1,
}

animal_choicer = Loot(animal_weights)

file = Path("mots.txt")

with open(file, "w", encoding="utf-8") as f:
    for _ in range(500):
        f.write(animal_choicer.next() + "\n")

print("File created :", file.resolve())
print("File content :")
print(file.read_text(encoding="utf-8"))

File created : /content/mots.txt
File content :
lapin
chat
chat
chat
ours
chien
pigeon
loup
écureuil
chien
lapin
chien
chien
ours
souris
chat
lapin
chien
chien
pigeon
lapin
chien
pigeon
chat
chat
souris
chouette
chien
lapin
chien
pigeon
aigle
pigeon
chien
souris
lapin
chien
sanglier
chat
écureuil
chien
souris
pigeon
chien
écureuil
pigeon
loup
souris
écureuil
chien
chien
écureuil
lapin
écureuil
souris
pigeon
pigeon
chien
souris
cerf
chat
écureuil
pigeon
souris
lapin
pigeon
pigeon
souris
chat
pigeon
souris
chat
aigle
souris
écureuil
pigeon
écureuil
pigeon
pigeon
lapin
écureuil
cerf
souris
écureuil
souris
chien
chat
chat
chouette
pigeon
lapin
souris
pigeon
souris
loup
écureuil
chien
chien
chien
écureuil
souris
écureuil
chat
souris
souris
pigeon
pigeon
renard
écureuil
pigeon
sanglier
chouette
écureuil
écureuil
chat
sanglier
pigeon
ours
pigeon
loup
chien
écureuil
renard
chien
pigeon
lapin
renard
loup
souris
souris
chien
chat
renard
cerf
chien
lapin
aigle
lapin
chien
écureuil
souris
chat
cha

In [3]:
from time import perf_counter

def execute_with_performances(function) :
    s = perf_counter()
    r = function()
    t = perf_counter() - s
    return r, t


In [4]:
from pathlib import Path

def count_words_python(file_path):
    word_counts = {}

    with open(file_path, "r", encoding="utf-8") as file:
        for word in file.read().split():
            word_counts[word] = word_counts.get(word, 0) + 1

    return word_counts

python_results, python_time = execute_with_performances(
    lambda: count_words_python("mots.txt")
)

print("Python Word Counter")
print("Results:", python_results)
print(f"Execution time: {python_time:.6f} seconds")

Python Word Counter
Results: {'lapin': 39, 'chat': 51, 'ours': 4, 'chien': 57, 'pigeon': 84, 'loup': 10, 'écureuil': 67, 'souris': 97, 'chouette': 14, 'aigle': 9, 'sanglier': 14, 'cerf': 23, 'renard': 28, 'lynx': 2, 'panthère': 1}
Execution time: 0.002536 seconds


In [5]:

def map_words(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return [(word, 1) for word in file.read().split()]

def shuffle_words(mapped_data):
    grouped_data = {}

    for word, count in mapped_data:
        grouped_data.setdefault(word, []).append(count)

    return grouped_data

def reduce_words(grouped_data):
    return {
        word: sum(counts)
        for word, counts in grouped_data.items()
    }

def count_words_mapreduce(file_path):
    mapped_data = map_words(file_path)
    shuffled_data = shuffle_words(mapped_data)
    return reduce_words(shuffled_data)

mapreduce_results, mapreduce_time = execute_with_performances(
    lambda: count_words_mapreduce("mots.txt")
)

print("Pseudo-MapReduce Word Counter")
print("Results:", mapreduce_results)
print(f"Execution time: {mapreduce_time:.6f} seconds")

Pseudo-MapReduce Word Counter
Results: {'lapin': 39, 'chat': 51, 'ours': 4, 'chien': 57, 'pigeon': 84, 'loup': 10, 'écureuil': 67, 'souris': 97, 'chouette': 14, 'aigle': 9, 'sanglier': 14, 'cerf': 23, 'renard': 28, 'lynx': 2, 'panthère': 1}
Execution time: 0.000397 seconds


In [6]:

%%writefile mapper.py
import sys

for line in sys.stdin:
    for word in line.strip().split():
        print(f"{word}\t1")


Writing mapper.py


In [7]:

%%writefile reducer.py
import sys

current_word = None
current_count = 0

for line in sys.stdin:
    word, count = line.strip().split("\t", 1)
    count = int(count)

    if word == current_word:
        current_count += count
    else:
        if current_word is not None:
            print(f"{current_word}\t{current_count}")

        current_word = word
        current_count = count

if current_word is not None:
    print(f"{current_word}\t{current_count}")


Writing reducer.py


In [ ]:

import os
import subprocess
import shutil
from pathlib import Path

# Configuration de Hadoop pour le noyau du notebook.
hadoop_home = Path("/home/dada/Documents/Cours/BigData/hadoop-3.5.0")
if not (hadoop_home / "bin/hadoop").is_file():
    raise FileNotFoundError(f"Hadoop introuvable dans {hadoop_home}")

java = shutil.which("java")
if java is None:
    raise RuntimeError("Java est introuvable dans le PATH")

os.environ["HADOOP_HOME"] = str(hadoop_home)
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)
os.environ["PATH"] = f"{hadoop_home / 'bin'}:{os.environ['PATH']}"

subprocess.run(["hadoop", "version"], check=True)

def count_words_hadoop(file_path):
    streaming_jars = list(Path(os.environ["HADOOP_HOME"]).glob(
        "share/hadoop/tools/lib/hadoop-streaming*.jar"
    ))

    if not streaming_jars:
        raise FileNotFoundError("Hadoop Streaming JAR not found")

    streaming_jar = str(streaming_jars[0])
    input_path = str(Path("wordcount/input").resolve())
    output_path = str(Path("wordcount/output").resolve())

    subprocess.run(
        ["hdfs", "dfs", "-mkdir", "-p", input_path],
        check=True
    )

    subprocess.run(
        ["hdfs", "dfs", "-put", "-f", file_path, input_path],
        check=True
    )

    subprocess.run(
        ["hdfs", "dfs", "-rm", "-r", "-f", output_path],
        check=True,
        capture_output=True
    )

    subprocess.run(
        [
            "hadoop", "jar", streaming_jar,
            "-files", "mapper.py,reducer.py",
            "-input", f"{input_path}/{Path(file_path).name}",
            "-output", output_path,
            "-mapper", "python3 mapper.py",
            "-reducer", "python3 reducer.py"
        ],
        check=True
    )

    result = subprocess.run(
        ["hdfs", "dfs", "-cat", f"{output_path}/part-*"],
        capture_output=True,
        text=True,
        check=True
    )

    return {
        word: int(count)
        for word, count in (
            line.split("\t") for line in result.stdout.splitlines()
        )
    }

hadoop_results, hadoop_time = execute_with_performances(
    lambda: count_words_hadoop("mots.txt")
)

print("Hadoop MapReduce Word Counter")
print("Results:", hadoop_results)
print(f"Execution time: {hadoop_time:.6f} seconds")


In [ ]:

import pandas as pd

assert python_results == mapreduce_results == hadoop_results

comparison = pd.DataFrame({
    "Method": ["Plain Python", "Pseudo-MapReduce", "Hadoop MapReduce"],
    "Execution Time (s)": [python_time, mapreduce_time, hadoop_time],
    "Unique Words": [
        len(python_results),
        len(mapreduce_results),
        len(hadoop_results)
    ],
    "Total Words": [
        sum(python_results.values()),
        sum(mapreduce_results.values()),
        sum(hadoop_results.values())
    ]
})

display(comparison)
